# 실습 B: 데이터 수집 자동화 — OutputParser 집중 실습

LLM의 자유 형식 텍스트 응답을 **파이썬 객체**로 바꾸는 것, 이것이 OutputParser의 핵심 역할이에요.<br/>
이 노트북에서는 뉴스 기사 파싱부터 오류 자동 복구까지, 실무에서 바로 쓸 수 있는 5가지 시나리오를 직접 구현해봐요.

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:
- `PydanticOutputParser`로 텍스트를 Pydantic 모델 객체로 변환할 수 있어요
- `JsonOutputParser`로 구조화된 JSON을 생성하고 `batch()`로 여러 입력을 한 번에 처리할 수 있어요
- `CommaSeparatedListOutputParser`로 키워드를 추출하고 빈도를 집계할 수 있어요
- `OutputFixingParser`로 파싱 오류를 자동으로 복구할 수 있어요
- `with_structured_output()`과 `PydanticOutputParser`의 차이를 설명하고 선택할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 Ch03 OutputParser 시리즈(노트북 01~06)에서 다룬 각 파서의 기본 사용법을 이해하고 있어야 해요.

## 실습 전체 구성

아래 다이어그램은 5가지 실습의 흐름과 각각 사용하는 파서를 한눈에 보여줘요.

```mermaid
flowchart TD
    subgraph P1["실습 1: 뉴스 기사 파서"]
        A1["뉴스 텍스트"]:::input --> B1["PydanticOutputParser"]:::process --> C1["NewsArticle 객체"]:::output
    end
    subgraph P2["실습 2: 제품 비교표"]
        A2["제품 설명 목록"]:::input --> B2["JsonOutputParser"]:::process --> C2["비교 JSON"]:::output
    end
    subgraph P3["실습 3: 설문 분석기"]
        A3["설문 응답"]:::input --> B3["CommaSeparatedList"]:::process --> C3["키워드 빈도"]:::output
    end
    subgraph P4["실습 4: 오류 복구"]
        A4["잘못된 형식"]:::input --> B4["OutputFixingParser"]:::process --> C4["복구된 객체"]:::output
        B4 -.->|"오류 발생"| E4["재요청"]:::error
        E4 --> B4
    end
    subgraph P5["실습 5 (보너스): 방식 비교"]
        A5["입력"]:::input --> B5a["PydanticOutputParser"]:::process
        A5 --> B5b["with_structured_output"]:::process
        B5a --> C5["동일한 결과"]:::output
        B5b --> C5
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
```

## 환경 설정

모든 실습에서 공통으로 사용하는 라이브러리와 모델을 초기화해요.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

# temperature=0: 구조화된 데이터 추출 시 일관된 출력을 위해 재현성 확보
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

---

## 실습 1: 뉴스 기사 파서

뉴스 텍스트에서 제목, 요약, 카테고리, 키워드를 자동으로 추출해볼게요.<br/>
`PydanticOutputParser`는 Pydantic 모델을 기반으로 LLM에 정확한 출력 형식을 지시하고,<br/>
응답을 타입이 검증된 객체로 변환해줘요.

### 과제

**요구사항**:
- `NewsArticle` Pydantic 모델을 정의하세요 (필드: `title`, `summary`, `category`, `keywords`)
- `PydanticOutputParser`와 `ChatPromptTemplate`으로 파싱 체인을 구성하세요
- 아래 뉴스 텍스트를 입력해서 `NewsArticle` 객체로 반환하세요

**힌트**: `parser.get_format_instructions()`로 형식 지침을 프롬프트에 주입하고,<br/>
체인 마지막에 `| parser`를 연결하세요.

In [24]:
# ---------------------------------------------------
# 실습 1에서 파싱할 뉴스 기사 텍스트
# ---------------------------------------------------

news_text = """
국내 AI 스타트업 테크비전이 자체 개발한 한국어 특화 언어 모델 '코어LM 2.0'을
오늘 공개했다. 이번 모델은 한국어 문서 2천억 토큰으로 학습됐으며, 법률·금융·의료
분야에서 기존 범용 모델 대비 정확도가 15% 향상됐다고 회사 측은 밝혔다.

테크비전은 코어LM 2.0을 API 형태로 제공할 예정이며, 내달부터 베타 서비스를 시작한다.
현재 금융권과 법률 플랫폼 기업 3곳이 파일럿 계약을 체결한 상태다. 회사 대표는
"국내 산업 현장에 맞는 AI를 만드는 것이 목표"라고 말했다.
"""

In [26]:
# ============================================================
# TODO: NewsArticle Pydantic 모델과 PydanticOutputParser를 정의하세요
# 힌트: BaseModel을 상속하고, Field(description=...) 로 각 필드 설명을 작성하세요
#       keywords 필드는 List[str] 타입으로 정의하세요
# 예상 결과: parser.get_format_instructions() 호출 시 JSON 스키마가 출력됨
# ============================================================

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# 1단계: Pydantic 모델 정의
# - title: 기사 제목
# - summary: 2-3문장 요약
# - category: 기사 카테고리 (예: IT, 경제, 사회)
# - keywords: 핵심 키워드 리스트
class NewsArticle(BaseModel):
    """뉴스 기사에서 추출한 정보"""
    
    title: str = Field(description="기사 제목")
    summary: str = Field(description="2-3문장 요약")
    category: str = Field(description="기사 카테고리 (예: IT, 경제, 사회)")
    keywords: List[str] = Field(description="핵심 키워드 리스트")

# 2단계: PydanticOutputParser 생성
# parser = PydanticOutputParser(pydantic_object=NewsArticle)
parser = PydanticOutputParser(pydantic_object=NewsArticle)

# 3단계: 형식 지침 확인
# print(parser.get_format_instructions())
print(parser.get_format_instructions())


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "뉴스 기사에서 추출한 정보", "properties": {"title": {"description": "기사 제목", "title": "Title", "type": "string"}, "summary": {"description": "2-3문장 요약", "title": "Summary", "type": "string"}, "category": {"description": "기사 카테고리 (예: IT, 경제, 사회)", "title": "Category", "type": "string"}, "keywords": {"description": "핵심 키워드 리스트", "items": {"type": "string"}, "title": "Keywords", "type": "array"}}, "required": ["title", "summary", "category", "keywords"]}
```


모델과 파서를 정의했어요. 이제 프롬프트와 체인을 구성하고 실행해볼게요.

In [14]:
# ============================================================
# TODO: 파싱 체인을 구성하고 뉴스 텍스트를 NewsArticle 객체로 변환하세요
# 힌트: ChatPromptTemplate.from_messages()로 시스템/사용자 메시지를 구성하고
#       prompt.partial(format_instructions=parser.get_format_instructions())로
#       형식 지침을 프롬프트에 미리 주입하세요
# 예상 결과: NewsArticle(title=..., summary=..., category=..., keywords=[...])
# ============================================================

from langchain_core.prompts import ChatPromptTemplate

# 1단계: 프롬프트 구성
# - 시스템 메시지: 뉴스 분석 전문가 역할 부여
# - 사용자 메시지: {news_text}와 {format_instructions} 변수 포함

prompt = ChatPromptTemplate.from_messages([
  ("system", "너는 지금부터 뉴스 분석 전문가야 사용자가 입력한 뉴스를 분석하고 정보를 추출해"),
  ("human", "{news_text}\n\n{format_instructions}")
])

# 2단계: 형식 지침을 프롬프트에 주입 (partial)
prompt = prompt.partial(format_instructions=parser.get_format_instructions())


# 3단계: 체인 구성 (prompt | model | parser)
chain = (
  prompt
  | model
  | parser
)

# 4단계: 체인 실행 및 결과 출력
res = chain.invoke({"news_text": news_text})

res

NewsArticle(title="테크비전, 한국어 특화 언어 모델 '코어LM 2.0' 공개", summary="국내 AI 스타트업 테크비전이 한국어 특화 언어 모델 '코어LM 2.0'을 공개했다. 이 모델은 2천억 토큰으로 학습되었으며, 법률, 금융, 의료 분야에서 기존 모델보다 15% 높은 정확도를 자랑한다.", category='IT', keywords=['AI', '스타트업', '언어 모델', '코어LM 2.0', '테크비전', '한국어', '정확도 향상', 'API', '베타 서비스'])

### 풀이

In [ ]:
# ---------------------------------------------------
# 실습 1 풀이: PydanticOutputParser로 뉴스 기사 파싱
# ---------------------------------------------------
pass

In [ ]:
# 3단계: 프롬프트 구성
pass

---

## 실습 2: 제품 비교표 생성

여러 제품 설명 텍스트를 `JsonOutputParser`로 구조화하고,<br/>
`batch()`로 한 번에 여러 제품을 처리해볼게요.<br/>
`JsonOutputParser`는 Pydantic 없이도 사용할 수 있어서 빠른 프로토타이핑에 적합해요.

### 과제

**요구사항**:
- `JsonOutputParser`에 `ProductSpec` Pydantic 스키마를 적용하세요
- `ChatPromptTemplate`으로 제품 설명을 JSON으로 변환하는 체인을 구성하세요
- `chain.batch()`로 여러 제품을 한 번에 처리하세요

**힌트**: `JsonOutputParser(pydantic_object=ProductSpec)` 형태로 스키마를 지정하세요.<br/>
`batch()`는 `[{"product_desc": ...}, {"product_desc": ...}]` 형태의 리스트를 받아요.

In [19]:
# ---------------------------------------------------
# 실습 2에서 처리할 제품 설명 데이터
# ---------------------------------------------------

product_descriptions = [
    """
    맥북 에어 M3 (15인치): 애플 M3 칩 탑재, 18시간 배터리, 15.3인치 Liquid Retina 디스플레이.
    무게 1.51kg, 256GB SSD, 8GB 메모리. 가격 179만 원. 색상은 미드나이트, 스타라이트 두 가지.
    팬리스 설계로 완전 무소음 동작.
    """,
    """
    갤럭시북4 Pro 360: 인텔 코어 Ultra 7 탑재 2-in-1 노트북. 16인치 AMOLED 터치스크린,
    360도 회전 힌지로 태블릿 모드 전환 가능. 무게 1.66kg, 512GB NVMe SSD, 16GB LPDDR5X RAM.
    가격 219만 원. S펜 기본 포함. AI 기능 강화된 갤럭시 AI 탑재.
    """,
    """
    LG 그램 Pro 16: 인텔 코어 Ultra 5, 16인치 IPS 디스플레이 (2560x1600). 무게 1.19kg으로
    초경량 설계. 512GB SSD, 16GB RAM. 배터리 80Wh (최대 22시간). 가격 189만 원.
    군수 규격(MIL-STD-810H) 내구성 인증. 지문인식·IR 카메라 탑재.
    """,
]

In [26]:
# ============================================================
# TODO: JsonOutputParser 스키마와 체인을 정의하세요
# 힌트: ProductSpec BaseModel에 name, price_krw, weight_kg, storage_gb,
#       memory_gb, display_inches, battery_hours, key_features(List[str]) 필드를 추가하세요
# 예상 결과: JsonOutputParser(pydantic_object=ProductSpec) 생성 후
#            parser.get_format_instructions() 출력 시 JSON 스키마가 보임
# ============================================================

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

# 1단계: JSON 출력 스키마 정의 (Pydantic 모델)
# - name: 제품명
# - price_krw: 가격 (원)
# - weight_kg: 무게 (kg)
# - storage_gb: 저장 용량 (GB)
# - memory_gb: 메모리 (GB)
# - display_inches: 화면 크기 (인치)
# - battery_hours: 배터리 지속 시간 (시간)
# - key_features: 주요 특징 (2-3개)
class ProductSpec(BaseModel):
    """제품 정보"""
    
    name: str = Field(description="제품명")
    price_krw: str = Field(description="가격 (원)")
    weight_kg: List[str] = Field(description="무게 (kg)")
    storage_gb: int = Field(description="저장 용량 (GB)")
    memory_gb: int = Field(description="메모리 (GB)")
    display_inches: int = Field(description="화면 크기 (인치)")
    battery_hours: int = Field(description="배터리 지속 시간 (시간)")
    key_features: int = Field(description="주요 특징 (2-3개)")

# 2단계: JsonOutputParser 생성
json_parser = JsonOutputParser(pydantic_object=ProductSpec)

# 3단계: 프롬프트 구성 + 형식 지침 주입
json_prompt = ChatPromptTemplate.from_messages([
  ("system", "다음 제품 설명을 읽고 제품 정보를 분석하시오"),
  ("human", "{product_desc}\n\n{format_instructions}")
])

json_prompt = json_prompt.partial(format_instructions=json_parser.get_format_instructions())

# 4단계: 체인 구성 (prompt | model | parser)
chain = (
  json_prompt
  | model
  | json_parser
)

In [27]:
# ============================================================
# TODO: batch()로 3개 제품을 한 번에 처리하고 비교표를 출력하세요
# 힌트: chain.batch([{"product_desc": desc} for desc in product_descriptions])
# 예상 결과: 3개의 dict가 담긴 리스트 반환, 각 dict에 제품 스펙 정보 포함
# ============================================================

# 1단계: batch()로 일괄 처리
# batch(): 여러 입력을 동시에 처리하여 invoke()를 반복 호출하는 것보다 효율적
ress = chain.batch([{"product_desc": desc} for desc in product_descriptions])

# 2단계: 비교표 출력
for i in ress:
    print(i)

{'name': '맥북 에어 M3 (15인치)', 'price_krw': '1790000', 'weight_kg': ['1.51'], 'storage_gb': 256, 'memory_gb': 8, 'display_inches': 15, 'battery_hours': 18, 'key_features': 3}
{'name': '갤럭시북4 Pro 360', 'price_krw': '2190000', 'weight_kg': ['1.66'], 'storage_gb': 512, 'memory_gb': 16, 'display_inches': 16, 'battery_hours': None, 'key_features': 3}
{'name': 'LG 그램 Pro 16', 'price_krw': '1890000', 'weight_kg': ['1.19'], 'storage_gb': 512, 'memory_gb': 16, 'display_inches': 16, 'battery_hours': 22, 'key_features': 3}


### 풀이

In [ ]:
# ---------------------------------------------------
# 실습 2 풀이: JsonOutputParser + batch() 제품 비교표
# ---------------------------------------------------
pass

In [ ]:
# ---------------------------------------------------
# batch()로 3개 제품 동시 처리 및 비교표 출력
# ---------------------------------------------------
pass

---

## 실습 3: 설문 응답 분석기

자유 형식 설문 응답에서 `CommaSeparatedListOutputParser`로 핵심 키워드를 추출하고,<br/>
여러 응답에 걸쳐 키워드 빈도를 집계해볼게요.<br/>
`CommaSeparatedListOutputParser`는 JSON 없이 가볍게 목록을 뽑아낼 때 유용해요.

### 과제

**요구사항**:
- `CommaSeparatedListOutputParser`로 각 응답에서 키워드를 추출하는 체인을 구성하세요
- 6개 응답에서 추출한 키워드 전체를 대상으로 빈도를 집계하세요
- 빈도가 높은 키워드 5개를 출력하세요

**힌트**: `CommaSeparatedListOutputParser.get_format_instructions()`를 프롬프트에 주입하세요.<br/>
`collections.Counter`로 빈도를 손쉽게 집계할 수 있어요.

In [3]:
# ---------------------------------------------------
# 실습 3에서 분석할 설문 응답 데이터
# ---------------------------------------------------

survey_responses = [
    "재택근무를 하면서 업무와 개인 생활의 경계가 모호해져서 스트레스가 늘었어요. 집중도 잘 안 되고 협업도 어렵더라고요.",
    "재택근무 덕분에 출퇴근 시간이 없어져서 개인 시간이 늘었어요. 다만 팀원들과의 소통이 줄어서 고립감을 느낄 때가 있어요.",
    "화상 회의가 많아지면서 오히려 피로도가 올라갔어요. 집에서 일하니까 쉬는 시간에도 계속 일하게 되는 경향이 있어요.",
    "재택근무 초반에는 적응이 힘들었지만 지금은 생산성이 높아진 것 같아요. 집중할 수 있는 환경이 만들어졌어요.",
    "아이를 돌보면서 일해야 해서 집중하기 어렵고 스트레스를 많이 받아요. 별도 업무 공간이 필요하다고 느껴요.",
    "팀원들과의 자연스러운 대화가 줄어서 협업이 어렵고 소속감도 떨어졌어요. 월 1회 이상 대면 미팅이 필요한 것 같아요.",
]

In [5]:
# ============================================================
# TODO: CommaSeparatedListOutputParser로 키워드 추출 체인을 구성하세요
# 힌트: output_parser.get_format_instructions()를 프롬프트에 partial()로 주입하고
#       chain.invoke({"response": ...})로 각 응답을 처리하세요
# 예상 결과: 각 응답마다 ['키워드1', '키워드2', ...] 형태의 리스트 반환
# ============================================================

from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1단계: CommaSeparatedListOutputParser 생성
# CommaSeparatedListOutputParser: 응답을 쉼표로 구분하여 파이썬 리스트로 반환
output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()
# 2단계: 프롬프트 구성 + 형식 지침 주입
# 시스템 메시지: 키워드 추출 전문가 역할 부여
# 사용자 메시지: 응답 텍스트 {response}와 {format_instructions} 포함
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 목록에서 핵심 키워드를 뽑아내시오."),
    ("human", "응답 목록 {response}\n{format_instructions}"),
])

prompt = prompt.partial(format_instructions=format_instructions)

# 3단계: 체인 구성 (prompt | model | output_parser)
chain = (
    prompt
    | model
    | output_parser
)

# 4단계: 모든 응답에서 키워드 추출
all_keywords = []
for i, resp in enumerate(survey_responses, 1):
    keywords = chain.invoke({"response": resp})
    all_keywords.extend(keywords)
    print(f"응답 {i}: {keywords}")
    pass

응답 1: ['재택근무', '업무', '개인 생활', '경계', '스트레스', '집중', '협업']
응답 2: ['재택근무', '출퇴근 시간', '개인 시간', '팀원', '소통', '고립감']
응답 3: ['화상 회의', '피로도', '집에서 일하기', '쉬는 시간', '일하는 경향']
응답 4: ['재택근무', '적응', '생산성', '집중', '환경']
응답 5: ['아이 돌봄', '일', '집중', '스트레스', '업무 공간 필요']
응답 6: ['팀원', '자연스러운 대화', '협업', '소속감', '대면 미팅', '월 1회 이상']


In [9]:
# ============================================================
# TODO: 추출된 키워드 빈도를 집계하고 상위 5개를 출력하세요
# 힌트: collections.Counter(all_keywords).most_common(5)를 활용하세요
# 예상 결과: 가장 많이 등장한 키워드와 등장 횟수가 순서대로 출력됨
# ============================================================

from collections import Counter

# 1단계: 빈도 집계
res = Counter(all_keywords).most_common(5)

# 2단계: 상위 5개 출력
res

[('재택근무', 3), ('집중', 3), ('스트레스', 2), ('협업', 2), ('팀원', 2)]

### 풀이

In [ ]:
# ---------------------------------------------------
# 실습 3 풀이: CommaSeparatedListOutputParser로 키워드 추출
# ---------------------------------------------------
pass

In [ ]:
# ---------------------------------------------------
# 빈도 집계 및 상위 키워드 출력
# ---------------------------------------------------
pass

---

## 실습 4: 오류 복구 파이프라인

실제 환경에서 LLM은 간혹 잘못된 형식으로 응답해요.<br/>
이 실습에서는 파싱 실패를 의도적으로 유발한 뒤, `OutputFixingParser`로 자동 복구하는 과정을 확인해봐요.<br/>
`OutputFixingParser`는 기본 파서 위에 LLM 재요청 레이어를 추가하는 래퍼(Wrapper) 패턴이에요.

```mermaid
flowchart LR
    A["잘못된 형식 문자열"]:::input -->|"parse() 시도"| B["PydanticOutputParser"]:::process
    B -->|"OutputParserException"| C["OutputFixingParser"]:::process
    C -->|"오류 + 원본 전달"| D["LLM<br/>(수정 요청)"]:::external
    D -->|"수정된 JSON"| B
    B -->|"파싱 성공"| E["Pydantic 객체"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

> **실무 팁**: `OutputFixingParser`는 오류 발생 시 LLM 호출을 한 번 더 해요.<br/>
> 추가 비용과 지연이 생기므로, 프롬프트 품질을 높여 오류 자체를 줄이는 것이 먼저예요.<br/>
> `OutputFixingParser`는 마지막 안전망으로 활용하세요.

### 과제

**요구사항**:
- `EventInfo` Pydantic 모델을 정의하세요
- 아래 3가지 잘못된 형식 데이터로 기본 파서가 실패하는 것을 확인하세요
- `OutputFixingParser`로 동일한 데이터를 자동 복구하세요

**힌트**: `OutputFixingParser.from_llm(parser=base_parser, llm=model)`로 생성하세요.

In [20]:
# ============================================================
# TODO: EventInfo 모델과 기본 파서를 정의하세요
# 힌트: EventInfo는 event_name(str), date(str), venue(str),
#       max_attendees(int), ticket_price_krw(int) 필드를 가져요
# 예상 결과: PydanticOutputParser 생성 완료 메시지 출력
# ============================================================

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1단계: 이벤트 정보 Pydantic 모델 정의
# - event_name: 이벤트 이름
# - date: 개최 날짜 (YYYY-MM-DD 형식)
# - venue: 개최 장소
# - max_attendees: 최대 참가 인원 (정수)
# - ticket_price_krw: 티켓 가격 (원, 정수)
class EventInfo(BaseModel):
    event_name: str = Field(description="이벤트 이름")
    date: str = Field(description="개최 날짜 (YYYY-MM-DD 형식)")
    venue: str = Field(description="개최 장소")
    max_attendees: str = Field(description="최대 참가 인원 (정수)")
    ticket_price_krw: str = Field(description="티켓 가격 (원, 정수)")

# 2단계: 기본 PydanticOutputParser 생성
base_parser = PydanticOutputParser(pydantic_object=EventInfo)

In [21]:
# ============================================================
# TODO: 잘못된 형식 데이터로 기본 파서 실패를 확인하세요
# 힌트: try-except로 각 데이터를 base_parser.parse()에 넣어보세요
# 예상 결과: 3건 모두 OutputParserException 또는 ValidationError 발생
# ============================================================

# 의도적으로 파싱 오류를 유발하는 3가지 잘못된 형식 데이터
bad_event_data = [
    # 1. Python dict 형식 (작은따옴표 → JSON 규격 위반)
    "{'event_name': '서울 AI 컨퍼런스', 'date': '2025-09-15', 'venue': 'COEX 그랜드볼룸', 'max_attendees': 500, 'ticket_price_krw': 50000}",
    # 2. 필드 누락 (ticket_price_krw 없음)
    '{"event_name": "부산 스타트업 위크", "date": "2025-10-20", "venue": "벡스코", "max_attendees": 1200}',
    # 3. 타입 불일치 (max_attendees가 숫자가 아닌 문자열)
    '{"event_name": "제주 테크 서밋", "date": "2025-11-05", "venue": "제주 ICC", "max_attendees": "약 삼백명", "ticket_price_krw": 30000}',
]

print("[기본 파서 파싱 시도]")
print("=" * 60)

for i, data in enumerate(bad_event_data, 1):
    print(f"\n데이터 {i}: {data[:60]}...")
    try:
        # 여기서 base_parser.parse(data)를 호출하세요
        base_parser.parse(data)
    except Exception as e:
        print(f"  실패: {type(e).__name__}")

[기본 파서 파싱 시도]

데이터 1: {'event_name': '서울 AI 컨퍼런스', 'date': '2025-09-15', 'venue': ...
  실패: OutputParserException

데이터 2: {"event_name": "부산 스타트업 위크", "date": "2025-10-20", "venue": ...
  실패: OutputParserException

데이터 3: {"event_name": "제주 테크 서밋", "date": "2025-11-05", "venue": "제...
  실패: OutputParserException


In [23]:
# ============================================================
# TODO: OutputFixingParser를 생성하고 동일한 데이터를 자동 복구하세요
# 힌트: OutputFixingParser.from_llm(parser=base_parser, llm=model)
#       fixing_parser.parse(data)로 각 데이터를 파싱하세요
# 예상 결과: 3건 모두 EventInfo 객체로 성공적으로 복구됨
# ============================================================

from langchain.output_parsers import OutputFixingParser

# 1단계: OutputFixingParser 생성
# from_llm(): 기본 파서를 감싸고, 오류 발생 시 LLM에 수정을 요청하는 래퍼 생성
fixing_parser = OutputFixingParser.from_llm(
    parser=base_parser,
    llm=model
)

# 2단계: 동일한 데이터를 OutputFixingParser로 파싱
print("[OutputFixingParser 자동 복구]")
print("=" * 60)

for i, data in enumerate(bad_event_data, 1):
    print(f"\n데이터 {i}: {data[:60]}...")
    try:
        # fixing_parser.parse(data) 호출 및 결과 출력
        res = fixing_parser.parse(data)
        print(f"res: {res}")
    except Exception as e:
        print(f"  복구 실패: {e}")

[OutputFixingParser 자동 복구]

데이터 1: {'event_name': '서울 AI 컨퍼런스', 'date': '2025-09-15', 'venue': ...
res: event_name='서울 AI 컨퍼런스' date='2025-09-15' venue='COEX 그랜드볼룸' max_attendees='500' ticket_price_krw='50000'

데이터 2: {"event_name": "부산 스타트업 위크", "date": "2025-10-20", "venue": ...
res: event_name='부산 스타트업 위크' date='2025-10-20' venue='벡스코' max_attendees='1200' ticket_price_krw='50000'

데이터 3: {"event_name": "제주 테크 서밋", "date": "2025-11-05", "venue": "제...
res: event_name='제주 테크 서밋' date='2025-11-05' venue='제주 ICC' max_attendees='300' ticket_price_krw='30000'


### 풀이

In [ ]:
# ---------------------------------------------------
# 실습 4 풀이: OutputFixingParser 오류 복구 파이프라인
# ---------------------------------------------------
pass

---

## 실습 5 (보너스): `with_structured_output()`으로 간소화

실습 1에서 사용한 `PydanticOutputParser` 방식을 `with_structured_output()`으로 재구현하고,<br/>
두 방식의 코드와 동작 차이를 직접 비교해봐요.<br/>
`with_structured_output()`은 형식 지침 주입 없이 모델이 직접 구조화 출력을 생성해요.

### 두 방식 비교

아래 다이어그램은 같은 결과를 얻는 두 가지 경로를 보여줘요.

```mermaid
flowchart LR
    A["뉴스 텍스트"]:::input --> B1["PydanticOutputParser 방식"]:::process
    A --> B2["with_structured_output() 방식"]:::process

    B1 --> C1["1. get_format_instructions() 생성"]:::process
    C1 --> D1["2. 프롬프트에 지침 주입"]:::process
    D1 --> E1["3. LLM 호출"]:::external
    E1 --> F1["4. parser.parse() 파싱"]:::process
    F1 --> G["NewsArticle 객체"]:::output

    B2 --> C2["1. llm.with_structured_output() 바인딩"]:::process
    C2 --> D2["2. LLM 호출 (내부 자동 처리)"]:::external
    D2 --> G

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

### 과제

**요구사항**:
- 실습 1에서 정의한 `NewsArticle` 모델을 그대로 사용하세요
- `model.with_structured_output(NewsArticle)`로 구조화 출력 모델을 생성하세요
- `PydanticOutputParser` 방식과 `with_structured_output()` 방식을 나란히 실행하고 결과를 비교하세요

**힌트**: `with_structured_output()`은 `stream()`을 지원하지 않아요.<br/>
`invoke()`로 호출하고, 프롬프트에 형식 지침을 별도로 주입하지 않아도 돼요.

In [27]:
# ============================================================
# TODO: PydanticOutputParser 방식으로 뉴스 파싱 (실습 1 복습)
# 힌트: 실습 1의 체인을 그대로 재사용하거나, 이 셀에서 다시 구성하세요
# 예상 결과: NewsArticle 객체 반환
# ============================================================

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

# NewsArticle은 실습 1에서 정의한 모델을 재사용해요
# (이미 실습 1 셀을 실행했다면 바로 사용 가능)

# 1단계: PydanticOutputParser 방식 체인 구성
# parser → prompt(partial) → chain (prompt | model | parser)
parser = PydanticOutputParser(pydantic_object=NewsArticle)
parser.get_format_instructions()
prompt = ChatPromptTemplate.from_messages([
  ("system", "너는 지금부터 뉴스 분석 전문가야 사용자가 입력한 뉴스를 분석하고 정보를 추출해"),
  ("human", "{news_text}\n\n{format_instructions}")
])
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = (
  prompt
  | model
  | parser
)
# 2단계: 실행 및 결과 출력

res = chain.invoke({"news_text": news_text})
res

NewsArticle(title="테크비전, 한국어 특화 언어 모델 '코어LM 2.0' 공개", summary="국내 AI 스타트업 테크비전이 한국어 특화 언어 모델 '코어LM 2.0'을 공개했다. 이 모델은 2천억 토큰으로 학습되었으며, 법률, 금융, 의료 분야에서 기존 모델보다 15% 높은 정확도를 자랑한다.", category='IT', keywords=['AI', '스타트업', '언어 모델', '코어LM 2.0', '테크비전', '한국어', '정확도 향상', 'API', '베타 서비스'])

In [28]:
# ============================================================
# TODO: with_structured_output() 방식으로 동일한 뉴스를 파싱하세요
# 힌트: structured_model = model.with_structured_output(NewsArticle)
#       형식 지침을 별도로 주입하지 않아도 돼요
# 예상 결과: 실습 1과 동일한 구조의 NewsArticle 객체 반환
# ============================================================

# 1단계: with_structured_output() 바인딩
# 모델 자체가 NewsArticle 스키마를 인식하고 맞는 형태로 응답해요
structured_model = model.with_structured_output(NewsArticle)

# 2단계: 프롬프트 구성 (형식 지침 없이 단순하게)
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 뉴스 분석 전문가야. 입력된 뉴스 텍스트에서 주요 정보를 정해진 형식에 맞춰 추출해."),
    ("human", "{news_text}")
])

# 3단계: 실행 및 결과 출력
chain = prompt | structured_model
res = chain.invoke({"news_text": news_text})
res

NewsArticle(title="국내 AI 스타트업 테크비전, 한국어 특화 언어 모델 '코어LM 2.0' 공개", summary="테크비전이 한국어 특화 언어 모델 '코어LM 2.0'을 공개했다. 이 모델은 2천억 토큰으로 학습되었으며, 법률, 금융, 의료 분야에서 기존 모델보다 15% 높은 정확도를 자랑한다. API 형태로 제공될 예정이며, 내달부터 베타 서비스가 시작된다.", category='IT', keywords=['AI', '언어 모델', '코어LM 2.0', '테크비전', '한국어', '정확도 향상'])

### 풀이

In [ ]:
# ---------------------------------------------------
# 실습 5 풀이: PydanticOutputParser vs with_structured_output() 비교
# ---------------------------------------------------
pass

In [29]:
# ---------------------------------------------------
# 두 방식의 특성 비교 요약
# ---------------------------------------------------

print("=" * 70)
print(f"{'비교 항목':<20} | {'PydanticOutputParser':^22} | {'with_structured_output':^22}")
print("-" * 70)

comparison = [
    ("형식 지침 주입",     "필요 (get_format_instructions)", "불필요 (자동 처리)"),
    ("스트리밍 지원",      "가능 (stream())",                "불가능"),
    ("OutputFixingParser", "감쌀 수 있음",                   "불가 (직접 통합)"),
    ("코드 복잡도",        "중간",                           "낮음"),
    ("오류 투명성",        "높음 (parse() 단계 명시)",       "낮음 (내부 처리)"),
    ("권장 상황",          "파이프라인 세밀 제어",           "빠른 프로토타이핑"),
]

for label, v1, v2 in comparison:
    print(f"{label:<20} | {v1:^22} | {v2:^22}")

print("=" * 70)

print()
print("> 실무 팁: 빠르게 시작할 때는 with_structured_output()을 사용하고,")
print(">          OutputFixingParser 연동이나 스트리밍이 필요하면 PydanticOutputParser를 선택하세요.")

비교 항목                |  PydanticOutputParser  | with_structured_output
----------------------------------------------------------------------
형식 지침 주입             | 필요 (get_format_instructions) |      불필요 (자동 처리)      
스트리밍 지원              |     가능 (stream())      |          불가능          
OutputFixingParser   |        감쌀 수 있음         |       불가 (직접 통합)      
코드 복잡도               |           중간           |           낮음          
오류 투명성               |   높음 (parse() 단계 명시)   |       낮음 (내부 처리)      
권장 상황                |      파이프라인 세밀 제어       |       빠른 프로토타이핑       

> 실무 팁: 빠르게 시작할 때는 with_structured_output()을 사용하고,
>          OutputFixingParser 연동이나 스트리밍이 필요하면 PydanticOutputParser를 선택하세요.


---

## 정리

이번 실습에서 배운 내용을 정리해볼게요:

- `PydanticOutputParser`: Pydantic 모델 기반의 타입 검증 파싱. `get_format_instructions()`로 LLM에 형식을 지시해요
- `JsonOutputParser`: 스키마 유무에 관계없이 JSON으로 출력. `batch()`와 함께 쓰면 여러 입력을 효율적으로 처리해요
- `CommaSeparatedListOutputParser`: JSON 없이 가벼운 목록 추출. 키워드 수집·빈도 분석에 잘 맞아요
- `OutputFixingParser`: 기존 파서를 감싸는 Wrapper. 파싱 실패 시 LLM에 수정을 요청해 자동 복구해요
- `with_structured_output()`: 프롬프트 없이 모델이 직접 구조화 출력을 생성. 빠른 프로토타이핑에 적합해요

다음 단계로는 **Ch04 Memory** 챕터에서 대화 히스토리를 관리하고,<br/>
파서와 메모리를 결합해 대화형 데이터 수집 시스템을 구현하는 방법을 배워요.